In [1]:
import jax
import subprocess

def get_gpu_memory_usage_mb():
    try:
        devices = jax.devices()
        if not devices or devices[0].platform != "gpu":
            return 0

        logical_id = devices[0].id  

        cuda_visible = os.environ.get("CUDA_VISIBLE_DEVICES", None)
        if cuda_visible:
            physical_ids = [int(x) for x in cuda_visible.split(",")]
            physical_id = physical_ids[logical_id]
        else:
            physical_id = logical_id

        result = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,nounits,noheader'],
            encoding='utf-8'
        )
        mem_values = [int(x) for x in result.strip().split('\n')]

        return mem_values[physical_id]
    except Exception as e:
        print("[get_gpu_memory_usage_mb] get_gpu_memory failed：", e)
        return 0


In [2]:
import os
# 选择GPU
os.environ['CUDA_VISIBLE_DEVICES'] = '5'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.chdir('/home/zheling/Character_Recognition')

print("当前工作目录：", os.getcwd())


当前工作目录： /home/zheling/Character_Recognition


In [3]:
import os
import math
import array
from typing import List
import numpy as np
from joblib import Parallel, delayed
from tqdm import tqdm
import time

import jax
import jax.numpy as jnp
import jax.random as jr
import equinox as eqx
import numpy as np
import optax


def compute_path_signature(path, depth=3):
    T, d = path.shape
    if T < 2:
        return np.zeros(sum(d**k for k in range(1, depth+1)))
    
    increments = np.diff(path, axis=0)
    signatures = []
    
    # 1-degree
    signatures.append(np.sum(increments, axis=0))
    
    if depth >= 2:
        sig_2 = np.zeros(d * d)
        cumsum = np.zeros(d)
        for t in range(len(increments)):
            for i in range(d):
                for j in range(d):
                    sig_2[i * d + j] += cumsum[i] * increments[t, j]
            cumsum += increments[t]
        signatures.append(sig_2)
    
    if depth >= 3:
        sig_3 = np.zeros(d * d * d)
        cumsum1 = np.zeros(d)
        cumsum2 = np.zeros((d, d))
        for t in range(len(increments)):
            for i in range(d):
                for j in range(d):
                    for k in range(d):
                        sig_3[i * d**2 + j * d + k] += cumsum2[i, j] * increments[t, k]
            for i in range(d):
                for j in range(d):
                    cumsum2[i, j] += cumsum1[i] * increments[t, j]
            cumsum1 += increments[t]
        signatures.append(sig_3)
        
    if depth >= 4:
        sig_4 = np.zeros(d * d * d * d)
        cumsum1 = np.zeros(d)
        cumsum2 = np.zeros((d, d))
        cumsum3 = np.zeros((d, d, d))
        for t in range(len(increments)):
            for i in range(d):
                for j in range(d):
                    for k in range(d):
                        for l in range(d):
                            sig_4[i * d**3 + j * d**2 + k * d + l] += cumsum3[i, j, k] * increments[t, l]
            for i in range(d):
                for j in range(d):
                    for k in range(d):
                        cumsum3[i, j, k] += cumsum2[i, j] * increments[t, k]
            for i in range(d):
                for j in range(d):
                    cumsum2[i, j] += cumsum1[i] * increments[t, j]
            cumsum1 += increments[t]
        signatures.append(sig_4)

    return np.concatenate(signatures)


class DyadicPathSignatures:
    def __init__(self, dyadic_levels: int, signature_level: int = 2, overlapping: bool = False) -> None:
        self._dyadic_levels = dyadic_levels
        self._signature_level = signature_level
        self._overlapping = overlapping

    def __call__(self, sample: np.ndarray) -> np.ndarray:
        batch_size = sample.shape[0]
        dyadic_pieces: List[List[np.ndarray]] = [[] for _ in range(batch_size)]
        
        for dyadic_level in range(self._dyadic_levels + 1):
            if self._overlapping:
                num_pieces = 2**(dyadic_level + 1) - 1
                if num_pieces > 1:
                    frames_per_piece = sample.shape[1] / (num_pieces - 1)
                else:
                    frames_per_piece = sample.shape[1]
            else:
                num_pieces = 2**dyadic_level
                frames_per_piece = sample.shape[1] / num_pieces

            for i in range(batch_size):
                for j in range(num_pieces):
                    start_frame = int(j * frames_per_piece / 2**self._overlapping)
                    end_frame = int(start_frame + frames_per_piece)
                    piece = sample[i, start_frame:end_frame].astype(np.float64)
                    
                    sig = compute_path_signature(piece, depth=self._signature_level)
                    dyadic_pieces[i].append(sig)
        
        result = []
        for pieces in dyadic_pieces:
            result.append(np.concatenate(pieces))
        return np.array(result)


def hanging_normalization(input_array, method='sc'):
    trajectory = input_array[:, :2]
    T = trajectory.shape[0]

    if method == 'sc':
        start_point = trajectory[0]
        center_point = trajectory.mean(axis=0)
    elif method == 'se':
        start_point = trajectory[0]
        center_point = trajectory[-1]
    else:
        raise ValueError("Invalid hanging normalization method. Use 'sc' or 'se'.")

    dx = center_point[0] - start_point[0]
    dy = center_point[1] - start_point[1]
    angle = np.arctan2(dy, dx) + np.pi / 2

    normalized_trajectory = np.zeros_like(trajectory)
    cos_a, sin_a = np.cos(-angle), np.sin(-angle)
    for t in range(T):
        x, y = trajectory[t]
        normalized_x = (x - start_point[0]) * cos_a - (y - start_point[1]) * sin_a
        normalized_y = (x - start_point[0]) * sin_a + (y - start_point[1]) * cos_a
        normalized_trajectory[t] = [normalized_x, normalized_y]

    output = input_array.copy()
    output[:, :2] = normalized_trajectory
    return output


def normalize_stroke(input_array, is_keep_ratio=True):
    stroke = input_array[:, :2]
    min_x, min_y = stroke.min(axis=0)
    max_x, max_y = stroke.max(axis=0)
    width = max_x - min_x
    height = max_y - min_y
    
    if is_keep_ratio:
        factor = max(width, height)
        if factor > 0:
            stroke[:, 0] = (stroke[:, 0] - min_x) / factor
            stroke[:, 1] = (stroke[:, 1] - min_y) / factor
    else:
        if width > 0:
            stroke[:, 0] = (stroke[:, 0] - min_x) / width
        if height > 0:
            stroke[:, 1] = (stroke[:, 1] - min_y) / height
    
    output = input_array.copy()
    output[:, :2] = stroke
    return output


def apply_rotation(trajectory, angle_degrees):
    angle_rad = np.radians(angle_degrees)
    cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
    rotation_matrix = np.array([[cos_a, -sin_a], [sin_a, cos_a]])
    
    coords = trajectory[:, :2]
    rotated_coords = coords @ rotation_matrix.T
    
    result = trajectory.copy()
    result[:, :2] = rotated_coords
    return result


def add_imaginary_strokes(strokes, enable_imaginary=True):
    if not enable_imaginary:
        return strokes
    
    total_segments = 0
    total_length = 0
    for stroke in strokes:
        if len(stroke) > 1:
            diffs = stroke[1:] - stroke[:-1]
            lengths = np.linalg.norm(diffs[:, :2], axis=1)
            total_length += np.sum(lengths)
            total_segments += len(lengths)
    
    if total_segments == 0:
        return strokes
    
    avg_segment_length = total_length / total_segments
    result_strokes = []
    
    for i in range(len(strokes)):
        result_strokes.append(strokes[i])
        if i < len(strokes) - 1:
            start_point = strokes[i][-1]
            end_point = strokes[i+1][0]
            virtual_length = np.linalg.norm(end_point[:2] - start_point[:2])
            num_segments = max(1, int(np.ceil(virtual_length / avg_segment_length)))
            
            virtual_stroke = []
            for j in range(num_segments + 1):
                alpha = j / num_segments
                virtual_point = start_point.copy()
                virtual_point[:2] = start_point[:2] * (1-alpha) + end_point[:2] * alpha
                if len(virtual_point) > 2:
                    virtual_point[2] = start_point[2]
                virtual_stroke.append(virtual_point)
            result_strokes.append(np.array(virtual_stroke))
    
    return result_strokes


def read_pot_file(filename, isFlipY=True):
    a = array.array('H')
    with open(filename, 'rb') as f:
        a.fromfile(f, os.path.getsize(filename) // a.itemsize)
    a.reverse()
    samples = []
    
    while len(a) > 0:
        _ = a.pop()
        a.pop()
        a.pop()
        num_strokes = a.pop()
        strokes = []
        
        for _ in range(num_strokes):
            stroke = []
            point = [a.pop(), a.pop()]
            while point != [65535, 0]:
                if isFlipY:
                    point = [point[0], -point[1]]
                stroke.append(point)
                point = [a.pop(), a.pop()]
            strokes.append(np.array(stroke, dtype=np.float32))
        a.pop()
        a.pop()
        samples.append(strokes)
    
    return samples


def interpolate_1d(x, target_length):
    old_length = len(x)
    old_indices = np.linspace(0, old_length - 1, old_length)
    new_indices = np.linspace(0, old_length - 1, target_length)
    return np.interp(new_indices, old_indices, x)


def interpolate_trajectory(trajectory, target_length):
    result = np.zeros((target_length, trajectory.shape[1]))
    for i in range(trajectory.shape[1]):
        result[:, i] = interpolate_1d(trajectory[:, i], target_length)
    return result


def process_class_(class_idx, data_dir, sig, min_npoint, angle, is_train=True,
                   concat_strokes=False, hang_normalize=True, is_add_ink_dim=True,
                   enable_imaginary=True, normalize_ink=True, master_seed=42):
    filename = os.path.join(data_dir, f"{class_idx}.pot")
    if not os.path.exists(filename):
        return np.empty((0, 0)), np.empty(0, dtype=np.int64)
    
    raw_samples = read_pot_file(filename)
    features, labels = [], []
    N = len(raw_samples)
    train_border = int(0.8 * N)
    samples_to_process = raw_samples[:train_border] if is_train else raw_samples[train_border:]

    for strokes in samples_to_process:
        strokes = [apply_rotation(st, angle) for st in strokes]
        
        if is_add_ink_dim:
            ink_released = 0
            for i, stroke in enumerate(strokes):
                ink_dim = np.arange(ink_released, ink_released + stroke.shape[0], dtype=np.float32)
                ink_released += stroke.shape[0] - 1
                strokes[i] = np.concatenate([stroke, ink_dim[:, None]], axis=-1)
        
        if enable_imaginary:
            strokes_imaginary = add_imaginary_strokes(strokes, enable_imaginary)
        else:
            strokes_imaginary = []

        def feature_extraction(stk_list):
            if is_add_ink_dim and normalize_ink:
                final_ink = max([stroke[-1, -1] for stroke in stk_list if len(stroke) > 0])
                if final_ink > 0:
                    for i in range(len(stk_list)):
                        stk_list[i] = stk_list[i].copy()
                        stk_list[i][:, -1] = stk_list[i][:, -1] / final_ink
            
            sample = np.concatenate(stk_list, axis=0) if concat_strokes and stk_list else stk_list[0] if stk_list else np.empty((0, 3))
            if sample.size == 0:
                return None
            
            onetraj = normalize_stroke(sample)
            target_len = min_npoint * onetraj.shape[0]
            onetraj = interpolate_trajectory(onetraj, target_len)
            
            if hang_normalize:
                normalized_onetraj = hanging_normalization(onetraj, method='sc')
            else:
                normalized_onetraj = onetraj
            normalized_onetraj = normalize_stroke(normalized_onetraj)
            
            fea = sig(normalized_onetraj[None, ...])
            return fea[0]

        feature = feature_extraction(strokes)
        if feature is None:
            continue
        
        if enable_imaginary and strokes_imaginary:
            feature_imaginary = feature_extraction(strokes_imaginary)
            if feature_imaginary is not None:
                feature = np.concatenate([feature, feature_imaginary], axis=-1)
        
        features.append(feature)
        labels.append(class_idx - 1)

    return (np.stack(features) if features else np.empty((0, 0)),
            np.array(labels) if labels else np.empty(0, dtype=np.int64))

class RadicalDataset:
    def __init__(self, 
                 data_dir,
                 num_classes,
                 concat_strokes=False,
                 hang_normalize=True,
                 is_add_ink_dim=True,
                 is_train=True,
                 feature_norm_factor=None,
                 enable_imaginary=True,
                 normalize_ink=True,
                 master_seed=42,
                 dyadic_levels=3,
                 signature_level=3,
                 overlapping=False,
                 min_point=None   
                 ):
        
        self.feature_norm_factor = feature_norm_factor
        

        self.sig = DyadicPathSignatures(
            dyadic_levels=dyadic_levels,
            signature_level=signature_level,
            overlapping=overlapping
        )
        
        if min_point is None:
            self.min_point = 2 ** dyadic_levels + 1
        else:
            self.min_point = min_point
        
        if is_train:
            np.random.seed(master_seed)
            tasks = [(cls, np.random.uniform(0, 360)) for cls in range(1, num_classes + 1)]
        else:
            tasks = [(cls, angle_idx * 12) for cls in range(1, num_classes + 1) for angle_idx in range(30)]
        
        print(f"{'Train dataset' if is_train else 'Test dataset'}: Totally {len(tasks)} tasks")
        
        results = Parallel(n_jobs=-1)(
            delayed(process_class_)(
                cls, data_dir, self.sig, self.min_point, ang, is_train,
                concat_strokes, hang_normalize, is_add_ink_dim,
                enable_imaginary, normalize_ink, master_seed
            )
            for cls, ang in tqdm(tasks, desc=f"{'Train dataset' if is_train else 'Test dataset'}")
        )
        
        valid_results = [(f, l) for f, l in results if len(f) > 0]
        if valid_results:
            self.features = np.concatenate([f for f, _ in valid_results], axis=0)
            self.labels = np.concatenate([l for _, l in valid_results], axis=0)
        else:
            self.features = np.empty((0, 0))
            self.labels = np.empty(0, dtype=np.int64)
        
        if len(self.features) > 0:
            self.feature_normalization()
            print(f"{'Train dataset' if is_train else 'Test dataset'} Features shape: {self.features.shape}, Labels shape: {self.labels.shape}")

    def feature_dim(self):
        return self.features.shape[1] if len(self.features) > 0 else 0

    def feature_normalization(self):
        if self.feature_norm_factor is None:
            max_val = np.max(np.abs(self.features), axis=0)
            max_val[max_val == 0] = 1
            self.feature_norm_factor = max_val
        self.features = self.features / self.feature_norm_factor


def get_dataloader(data_dir, num_classes, batch_size=32, **kwargs):
    dataset = RadicalDataset(data_dir, num_classes, **kwargs)
    X = jnp.array(dataset.features)
    y = jnp.array(dataset.labels)
    
    if kwargs.get("is_train", True):
        key = jr.PRNGKey(kwargs.get("master_seed", 42))
        perm = jr.permutation(key, len(y))
        X, y = X[perm], y[perm]
    
    class DataLoader:
        def __init__(self, X, y, batch_size):
            self.X, self.y, self.batch_size = X, y, batch_size
            self.num_batches = math.ceil(len(y) / batch_size)
        
        def __iter__(self):
            for i in range(self.num_batches):
                yield self.X[i*self.batch_size:(i+1)*self.batch_size], \
                      self.y[i*self.batch_size:(i+1)*self.batch_size]
    
    return DataLoader(X, y, batch_size), dataset.feature_dim(), dataset.feature_norm_factor


class RotationFreeOLHCR(eqx.Module):
    layers: list
    dropouts: list
    
    def __init__(self, input_size, num_classes, *, key):
        keys = jr.split(key, 6)
        self.layers = [
            eqx.nn.Linear(input_size, 128, key=keys[0]),
            eqx.nn.Linear(128, 256, key=keys[1]),
            eqx.nn.Linear(256, 384, key=keys[2]),
            eqx.nn.Linear(384, 512, key=keys[3]),
            eqx.nn.Linear(512, 640, key=keys[4]),
            eqx.nn.Linear(640, num_classes, key=keys[5])
        ]
        self.dropouts = [eqx.nn.Dropout(p=p) for p in [0.1, 0.2, 0.3, 0.4, 0.5]]
    
    def __call__(self, x, key, inference=False):
        keys = jr.split(key, 5)
        for i in range(5):
            x = jax.nn.relu(self.layers[i](x))
            x = self.dropouts[i](x, key=keys[i], inference=inference)
        return self.layers[5](x)



In [4]:
@eqx.filter_jit
def train_step(model, X, y, opt, opt_state, subkey):
    def loss_fn(m):
        preds = jax.vmap(lambda x: m(x, subkey, inference=False))(X)
        loss = optax.softmax_cross_entropy_with_integer_labels(preds, y).mean()
        acc = jnp.mean(jnp.argmax(preds, axis=-1) == y)
        return loss, acc
    (loss, acc), grads = eqx.filter_value_and_grad(loss_fn, has_aux=True)(model)
    updates, opt_state = opt.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss, acc  


def train_epoch(model, loader, opt, opt_state, key):
    losses, accs, steps = [], [], 0
    for X, y in loader:
        key, subkey = jr.split(key)
        model, opt_state, loss, acc = train_step(model, X, y, opt, opt_state, subkey)
        losses.append(float(np.array(loss))) 
        accs.append(float(np.array(acc)))
        steps += 1
    return model, opt_state, np.mean(losses), np.mean(accs), steps



def train_loop(model, loader, epochs, lr_fn, key):
    opt = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(lr_fn))
    opt_state = opt.init(eqx.filter(model, eqx.is_array))
    log, mems = {"epoch": [], "loss": [], "acc": [], "time": [], "mem": []}, []
    total_steps, t0_total = 0, time.time()

    for ep in range(epochs):
        t0_epoch = time.time()
        key, epoch_key = jr.split(key)
        model, opt_state, l, a, steps_in_epoch = train_epoch(model, loader, opt, opt_state, epoch_key)
        total_steps += steps_in_epoch
        cur_mem = get_gpu_memory_usage_mb()
        mems.append(cur_mem)
        epoch_time = time.time() - t0_epoch

        if (ep+1) % 10 == 0 or ep == 0:
            print(f"Epoch {ep+1}/{epochs} | Loss: {l:.4f} | Acc: {a:.2%} | Mem: {cur_mem:.2f}MB | Time: {epoch_time:.2f}s")

        log["epoch"].append(ep+1)
        log["loss"].append(l)
        log["acc"].append(a)
        log["time"].append(epoch_time)
        log["mem"].append(cur_mem)

    
    total_time = time.time() - t0_total
    avg_mem = float(np.mean(log["mem"])) if log["mem"] else 0.0
    time_per_1000_steps = (total_time / total_steps * 1000) if total_steps > 0 else 0.0

    stats = {
        "total_time": total_time,
        "total_steps": total_steps,
        "avg_mem": avg_mem,
        "time_per_1000_steps": time_per_1000_steps
    }
    print("\n=== Training Stats ===")
    print(f"Total Time: {total_time:.2f}s")
    print(f"Total Steps: {total_steps}")
    print(f"Time per 1000 steps: {time_per_1000_steps:.2f}s")
    print(f"Avg GPU Memory: {avg_mem:.2f}MB")
    
    return model, log, stats


In [5]:
import numpy as np
import jax.random as jr
import equinox as eqx
import optax
import time
import jax
import jax.numpy as jnp




def evaluate_accuracy(model, loader, key):
    correct, total = 0, 0
    for X, y in loader:
        key, subkey = jr.split(key)
        logits = jax.vmap(lambda x: model(x, subkey, inference=True))(X)
        preds = jnp.argmax(logits, axis=-1)
        correct += int(jnp.sum(preds == y))
        total += len(y)
    return correct / total


def run_single_model(train_loader, test_loader, input_dim, num_classes, seed, num_epochs=300, lr=1e-3):

    key = jr.PRNGKey(seed)
    key, model_key = jr.split(key)
    model = RotationFreeOLHCR(input_dim, num_classes, key=model_key)

    lr_fn = optax.constant_schedule(lr)
    model, log, stats = train_loop(model, train_loader, num_epochs, lr_fn, key)

    key, eval_key = jr.split(key)
    test_acc = evaluate_accuracy(model, test_loader, eval_key)

    print(f"[Seed {seed}] Final Test Accuracy: {test_acc*100:.2f}%")
    return model, test_acc


def ensemble_evaluate(models, test_loader, seed=42):

    key = jr.PRNGKey(seed)
    all_logits = []
    all_targets = None

    for model in models:
        logits_list, targets_list = [], []
        for X, y in test_loader:
            key, subkey = jr.split(key)
            logits = jax.vmap(lambda x: model(x, subkey, inference=True))(X)
            logits_list.append(np.array(logits))
            targets_list.append(np.array(y))

        logits_all = np.concatenate(logits_list, axis=0)
        targets_all = np.concatenate(targets_list, axis=0)

        all_logits.append(logits_all)
        if all_targets is None:
            all_targets = targets_all

    all_logits = np.stack(all_logits, axis=0)  # [S, N, C]
    targets = all_targets

    # soft 
    probs = np.exp(all_logits) / np.sum(np.exp(all_logits), axis=-1, keepdims=True)
    avg_probs = np.mean(probs, axis=0)
    soft_acc = np.mean(np.argmax(avg_probs, -1) == targets)

    # hard 
    seed_preds = np.argmax(all_logits, axis=-1)  # [S, N]
    hard_preds = []
    for i in range(seed_preds.shape[1]):
        votes = np.bincount(seed_preds[:, i])
        hard_preds.append(np.argmax(votes))
    hard_preds = np.array(hard_preds)
    hard_acc = np.mean(hard_preds == targets)

    return soft_acc, hard_acc





In [ ]:
if __name__ == "__main__":

    seed_list = [1011, 1012, 1013, 1014, 1015]
    num_classes = 26
    data_dir = "RotFreeData/EngUpper26" # EngUpper26 / Digits10 / Radicals52
    MASTER_SEED = 42
    batch_size = 100

    print("\nBuilding training data...")
    train_loader, train_feadim, train_norm_factor = get_dataloader(
        data_dir, num_classes, batch_size=batch_size,
        concat_strokes=True, hang_normalize=True,
        is_add_ink_dim=True, is_train=True,
        feature_norm_factor=None,
        enable_imaginary=True, normalize_ink=True,
        master_seed=MASTER_SEED,
        dyadic_levels=3,       
        signature_level=4,      
        overlapping=False,      
    )

    print("\nBuilding test data...")
    test_loader, test_feadim, _ = get_dataloader(
        data_dir, num_classes, batch_size=batch_size,
        concat_strokes=True, hang_normalize=True,
        is_add_ink_dim=True, is_train=False,
        feature_norm_factor=train_norm_factor,
        enable_imaginary=True, normalize_ink=True,
        master_seed=MASTER_SEED,
        dyadic_levels=3,       
        signature_level=4,      
        overlapping=False,       
    )

    assert train_feadim == test_feadim
    print(f"\nFeature dimension: {train_feadim}")
    print(f"Training samples: {len(train_loader.X)}")
    print(f"Test samples: {len(test_loader.X)}\n")

    models, test_accs = [], []
    for seed in seed_list:
        print(f"\n===== Seed {seed} training started =====\n")
        model, acc = run_single_model(train_loader, test_loader, train_feadim, num_classes,
                                      seed=seed, num_epochs=300)
        models.append(model)
        test_accs.append(acc)

    mean_acc = np.mean(test_accs)
    std_acc = np.std(test_accs)
    print("\n===== Single-Model Accuracy Statistics =====")
    print(f"Mean Test Accuracy: {mean_acc*100:.2f}%")
    print(f"Std Dev: {std_acc*100:.2f}%")
    print(f"Mean ± Std: {mean_acc*100:.2f}% ± {std_acc*100:.2f}%")

    soft_acc, hard_acc = ensemble_evaluate(models, test_loader, seed=MASTER_SEED)
    print("\n===== Ensemble Evaluation =====")
    print(f"Soft voting accuracy: {soft_acc*100:.2f}%")
    print(f"Hard voting accuracy: {hard_acc*100:.2f}%")

In [8]:
# EngUpper26


Building training data...
Train dataset: Totally 26 tasks


Train dataset: 100%|██████████████████████████████████████████████████████████████████| 26/26 [00:00<00:00, 9109.67it/s]
/home/zheling/anaconda3/envs/py39/lib/python3.9/site-packages/joblib/externals/loky/backend/fork_exec.py:38: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid = os.fork()


Train dataset Features shape: (6237, 3600), Labels shape: (6237,)

Building test data...
Test dataset: Totally 780 tasks


Test dataset: 100%|███████████████████████████████████████████████████████████████████| 780/780 [03:47<00:00,  3.43it/s]


Test dataset Features shape: (46800, 3600), Labels shape: (46800,)

Feature dimension: 3600
Training samples: 6237
Test samples: 46800


===== Seed 1011 training started =====

Epoch 1/300 | Loss: 2.6714 | Acc: 16.94% | Mem: 5574.00MB | Time: 5.34s
Epoch 10/300 | Loss: 0.3081 | Acc: 90.33% | Mem: 5574.00MB | Time: 0.36s
Epoch 20/300 | Loss: 0.1530 | Acc: 96.04% | Mem: 5574.00MB | Time: 0.37s
Epoch 30/300 | Loss: 0.1009 | Acc: 96.94% | Mem: 5574.00MB | Time: 0.36s
Epoch 40/300 | Loss: 0.0893 | Acc: 97.76% | Mem: 5574.00MB | Time: 0.36s
Epoch 50/300 | Loss: 0.0637 | Acc: 98.37% | Mem: 5574.00MB | Time: 0.37s
Epoch 60/300 | Loss: 0.0512 | Acc: 98.75% | Mem: 5574.00MB | Time: 0.38s
Epoch 70/300 | Loss: 0.0464 | Acc: 98.89% | Mem: 5574.00MB | Time: 0.37s
Epoch 80/300 | Loss: 0.0434 | Acc: 98.87% | Mem: 5574.00MB | Time: 0.36s
Epoch 90/300 | Loss: 0.0450 | Acc: 98.94% | Mem: 5574.00MB | Time: 0.36s
Epoch 100/300 | Loss: 0.0421 | Acc: 99.10% | Mem: 5574.00MB | Time: 0.37s
Epoch 110/300 | Loss

/tmp/ipykernel_1780483/1214018732.py:64: RuntimeWarning: overflow encountered in exp
  probs = np.exp(all_logits) / np.sum(np.exp(all_logits), axis=-1, keepdims=True)
/tmp/ipykernel_1780483/1214018732.py:64: RuntimeWarning: invalid value encountered in divide
  probs = np.exp(all_logits) / np.sum(np.exp(all_logits), axis=-1, keepdims=True)


In [7]:
 #  Digits10 


Building training data...
Train dataset: Totally 10 tasks


Train dataset: 100%|██████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 2841.09it/s]
/home/zheling/anaconda3/envs/py39/lib/python3.9/site-packages/joblib/externals/loky/backend/fork_exec.py:38: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid = os.fork()


Train dataset Features shape: (2396, 3600), Labels shape: (2396,)

Building test data...
Test dataset: Totally 300 tasks


Test dataset: 100%|███████████████████████████████████████████████████████████████████| 300/300 [00:50<00:00,  6.00it/s]


Test dataset Features shape: (18000, 3600), Labels shape: (18000,)

Feature dimension: 3600
Training samples: 2396
Test samples: 18000


===== Seed 1011 training started =====

Epoch 1/300 | Loss: 2.0612 | Acc: 20.66% | Mem: 5568.00MB | Time: 5.33s
Epoch 10/300 | Loss: 0.1173 | Acc: 96.41% | Mem: 5568.00MB | Time: 0.30s
Epoch 20/300 | Loss: 0.0292 | Acc: 99.16% | Mem: 5568.00MB | Time: 0.28s
Epoch 30/300 | Loss: 0.0291 | Acc: 99.37% | Mem: 5568.00MB | Time: 0.29s
Epoch 40/300 | Loss: 0.0204 | Acc: 99.58% | Mem: 5568.00MB | Time: 0.29s
Epoch 50/300 | Loss: 0.0369 | Acc: 99.33% | Mem: 5568.00MB | Time: 0.27s
Epoch 60/300 | Loss: 0.0102 | Acc: 99.79% | Mem: 5568.00MB | Time: 0.27s
Epoch 70/300 | Loss: 0.0090 | Acc: 99.67% | Mem: 5568.00MB | Time: 0.27s
Epoch 80/300 | Loss: 0.0227 | Acc: 99.50% | Mem: 5568.00MB | Time: 0.27s
Epoch 90/300 | Loss: 0.0082 | Acc: 99.79% | Mem: 5568.00MB | Time: 0.30s
Epoch 100/300 | Loss: 0.0181 | Acc: 99.58% | Mem: 5568.00MB | Time: 0.24s
Epoch 110/300 | Loss

/tmp/ipykernel_1780483/1214018732.py:64: RuntimeWarning: overflow encountered in exp
  probs = np.exp(all_logits) / np.sum(np.exp(all_logits), axis=-1, keepdims=True)
/tmp/ipykernel_1780483/1214018732.py:64: RuntimeWarning: invalid value encountered in divide
  probs = np.exp(all_logits) / np.sum(np.exp(all_logits), axis=-1, keepdims=True)


In [6]:
# Radicals52


Building training data...
Train dataset: Totally 52 tasks


Train dataset: 100%|███████████████████████████████████████████████████████████████████| 52/52 [00:00<00:00, 984.93it/s]


Train dataset Features shape: (12401, 3600), Labels shape: (12401,)


2025-12-16 16:22:06.077453: W external/xla/xla/service/gpu/nvptx_compiler.cc:765] The NVIDIA driver's CUDA version is 12.2 which is older than the ptxas CUDA version (12.8.93). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.



Building test data...
Test dataset: Totally 1560 tasks


Test dataset: 100%|█████████████████████████████████████████████████████████████████| 1560/1560 [09:53<00:00,  2.63it/s]


Test dataset Features shape: (93510, 3600), Labels shape: (93510,)

Feature dimension: 3600
Training samples: 12401
Test samples: 93510


===== Seed 1011 training started =====

Epoch 1/300 | Loss: 3.0697 | Acc: 15.14% | Mem: 5564.00MB | Time: 6.02s
Epoch 10/300 | Loss: 0.4381 | Acc: 86.58% | Mem: 5566.00MB | Time: 0.49s
Epoch 20/300 | Loss: 0.2587 | Acc: 91.97% | Mem: 5566.00MB | Time: 0.47s
Epoch 30/300 | Loss: 0.1903 | Acc: 94.10% | Mem: 5566.00MB | Time: 0.47s
Epoch 40/300 | Loss: 0.1734 | Acc: 94.84% | Mem: 5566.00MB | Time: 0.48s
Epoch 50/300 | Loss: 0.1445 | Acc: 95.82% | Mem: 5566.00MB | Time: 0.54s
Epoch 60/300 | Loss: 0.1316 | Acc: 96.38% | Mem: 5566.00MB | Time: 0.47s
Epoch 70/300 | Loss: 0.1049 | Acc: 96.78% | Mem: 5566.00MB | Time: 0.49s
Epoch 80/300 | Loss: 0.1090 | Acc: 96.86% | Mem: 5566.00MB | Time: 0.46s
Epoch 90/300 | Loss: 0.1071 | Acc: 96.98% | Mem: 5566.00MB | Time: 0.46s
Epoch 100/300 | Loss: 0.0967 | Acc: 97.30% | Mem: 5566.00MB | Time: 0.46s
Epoch 110/300 | Los

/tmp/ipykernel_1780483/1214018732.py:64: RuntimeWarning: overflow encountered in exp
  probs = np.exp(all_logits) / np.sum(np.exp(all_logits), axis=-1, keepdims=True)
/tmp/ipykernel_1780483/1214018732.py:64: RuntimeWarning: invalid value encountered in divide
  probs = np.exp(all_logits) / np.sum(np.exp(all_logits), axis=-1, keepdims=True)



===== Ensemble Evaluation =====
Soft voting accuracy: 89.99%
Hard voting accuracy: 90.05%
